# Recommended beam-beam simulation configuration

Enter beam parameters, get the configuration recommended by the scaling laws: the vertical cell
count `n_y` and the macroparticle count `n_m`, at the transverse and longitudinal resolutions
the study fixed.

The recipe, the constants and their provenance are documented in `recommend.py`. In short: the
beam sizes give the vertical disruption `D_y`, `D_y` gives the depth of the pinch, and the grid
must resolve the pinched core and hold enough macroparticles inside its central cell.

**What this does not do.** It does not run a simulation, and it does not check convergence for
you. The laws were established on one machine (C3-250) over `eps_y = 1-20 nm`, at one machine
geometry (`beta_y*/sigma_z = 1.2`), for flat beams with head-on collisions. Anything outside
that is reported as a warning, and the `model` mode is an untested prediction.

In [1]:
from recommend import recommend, selftest, REFERENCE, D_Y_TESTED, HOURGLASS_REF

selftest()

R(D_y) reproduces the reference values to 5.1e-04
both recommended-configuration tables reproduced


True

## A single configuration

The reference machine, C3-250, at `eps_y = 8 nm`, for both codes.

In [2]:
beam = {k: v for k, v in REFERENCE.items() if k != 'name'}

for code in ('GP', 'WX'):
    print(recommend(eps_y_nm=8.0, code=code, **beam), end='\n\n')

GUINEA-PIG++ [recommended]: (n_x, n_y, n_z) = (512, 256, 64), n_t = 6, n_m = 223,511
    D_y = 34.216, D_x = 0.3226, beta_y*/sigma_z = 1.200, sigma_x* = 210.12 nm, sigma_y* = 1.981 nm
    cut multipliers (c_x, c_y, c_z) = (20, 20, 3.5)

WarpX [recommended]: (n_x, n_y, n_z) = (512, 256, 128), n_t = 128, n_m = 46,000
    D_y = 34.216, D_x = 0.3226, beta_y*/sigma_z = 1.200, sigma_x* = 210.12 nm, sigma_y* = 1.981 nm
    cut multipliers (c_x, c_y, c_z) = (16, 16, 8)



## A scan over emittance

This reproduces the recommended-configuration tables of the paper.

In [3]:
print(f"{'eps_y [nm]':>10} {'D_y':>7} | {'GP n_y':>7} {'GP n_m':>12} | {'WX n_y':>7} {'WX n_m':>10}")
for eps in (1.0, 2.0, 4.0, 8.0, 12.0, 16.0, 20.0):
    g = recommend(eps_y_nm=eps, code='GP', **beam)
    w = recommend(eps_y_nm=eps, code='WX', **beam)
    print(f'{eps:10g} {g.D_y:7.1f} | {g.n_y:7d} {g.n_m:12,} | {w.n_y:7d} {w.n_m:10,}')

eps_y [nm]     D_y |  GP n_y       GP n_m |  WX n_y     WX n_m
         1    97.4 |     512   14,096,339 |     512  1,400,000
         2    68.8 |     512    4,471,323 |     256    450,000
         4    48.5 |     256      707,822 |     256    140,000
         8    34.2 |     256      223,511 |     256     46,000
        12    27.9 |     256      113,696 |     256     23,000
        16    24.1 |     256       70,318 |     256     14,000
        20    21.5 |     256       48,411 |     256     10,000


/var/folders/68/5n6115cs1_s16kcr67xssx040000gn/T/ipykernel_49105/2595719273.py:3: OutsideTestedRange: n_m = 14,096,339 exceeds the largest count reached within the three-day wall-clock limit of this study.
  g = recommend(eps_y_nm=eps, code='GP', **beam)


## Outside what was covered

Warnings are raised as Python warnings and listed on the result. Here the emittance is below the
tuned range and the machine geometry is not the one the study ran, so the `recommended` constants
do not apply; `model` transports them with the envelope model and says so.

In [4]:
import warnings

other_machine = dict(beam, beta_y_m=4 * beam['beta_y_m'])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    r = recommend(eps_y_nm=0.4, code='WX', mode='model', **other_machine)
print(r)
print()
for c in caught:
    print(f'{c.category.__name__}: {c.message}')

WarpX [model]: (n_x, n_y, n_z) = (512, 1024, 128), n_t = 128, n_m = 420,000
    *** EXPERIMENTAL: transported to a machine geometry the study did not run ***
    D_y = 76.905, D_x = 0.3243, beta_y*/sigma_z = 4.800, sigma_x* = 210.12 nm, sigma_y* = 0.886 nm
    cut multipliers (c_x, c_y, c_z) = (16, 16, 8)
    compression R: 4.742 at beta_y*/sigma_z = 1.2 -> 14.439 here
    ! EXPERIMENTAL: beta_y*/sigma_z = 4.800 differs from the 1.2 the study ran, and the configuration has been transported by the envelope model. This is a prediction of the model, not a tested result.
    ! eps_y = 0.4 nm is outside the tuned range 1-20 nm.

ExperimentalRecommendation: EXPERIMENTAL: beta_y*/sigma_z = 4.800 differs from the 1.2 the study ran, and the configuration has been transported by the envelope model. This is a prediction of the model, not a tested result.
OutsideTestedRange: eps_y = 0.4 nm is outside the tuned range 1-20 nm.


A beam that is not flat is refused outright: the laws describe the vertical plane of a flat beam,
and on a tall beam the pinch is horizontal, so the vertical requirement would be the wrong answer.

In [5]:
try:
    recommend(eps_y_nm=900.0, code='GP', **dict(beam, eps_x_nm=2.0))
except ValueError as e:
    print('ValueError:', e)

ValueError: sigma_y* = 21.012 nm is not smaller than sigma_x* = 9.905 nm. These laws are derived for a flat beam, where the vertical plane is the strongly disrupted one; here the pinch would be horizontal, so the vertical requirement returned would apply to the wrong plane.
